# Lab L3.4 · Churn early-warning — logistic vs k-NN vs tree vs forest, value-based targeting, and trees for regression
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 3 · ⏱ 60 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Run a fair bake-off of every Day 3 algorithm on 250k customers, control tree depth, turn scores into a retention campaign with gains and expected value, and see what trees do (and cannot do) on a regression problem.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

About **12% of customers leave within six months**. The Retail Retention team can call 10% of the base per half-year and offers a fee waiver worth ≈ AED 150 per accepted offer. They ask MIZAN: *who should we call?* You will choose the algorithm on evidence, explain why depth matters for trees, and turn probabilities into a campaign list that maximises **net value**, not accuracy. The Mortgage team then asks a side question: can trees replace the Day 2 linear valuation model?

**Progressive architecture:** combines L1.1 (master), L1.3 (transaction features) and L3.2 (segments). The churn model is upgraded with a neural network in L4.3 and deployed in the Day 5 capstone (L5.3).

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Assemble, guard against leakage, split

**Why this matters.** Three earlier labs contribute features. Before modelling we assert that **leaky** columns (the account-closure date, retention offers made *after* the customer started leaving) and **protected attributes** are not in the feature list, then make a stratified 60/20/20 split so every model is judged on the same validation and test rows.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — split the remaining 40% into two equal halves

In [ ]:
import json
cust = pd.read_parquet(CKPT / "customers_clean.parquet")                    # L1.1
seg = pd.read_parquet(CKPT / "customer_segments.parquet")                  # L3.2
txf = pd.read_parquet(CKPT / "customer_txn_features.parquet")              # L1.3
TXN = ["has_card_activity", "txn_count", "log_amt_mean", "recency_days", "spend_trend", "ecom_share", "n_mcc"]
df = (cust.merge(seg[["customer_id", "segment"]], on="customer_id", validate="one_to_one")
          .merge(txf[["customer_id"] + TXN], on="customer_id", validate="one_to_one"))
df["log_balance"], df["log_salary"] = np.log1p(df.avg_balance_aed), np.log1p(df.monthly_salary_aed)
NUM = ["age", "tenure_months", "num_products", "salary_transfer", "has_credit_card", "has_personal_loan", "has_auto_loan", "has_mortgage",
       "log_balance", "log_salary", "digital_logins_30d", "branch_visits_90d", "complaints_12m", "days_since_last_txn", "aecb_score"] + TXN
CAT = ["segment", "employment_type", "emirate"]
FORBIDDEN = {"account_closed_date", "retention_offer_made", "gender", "nationality_group"}   # leakage + protected attributes
from sklearn.model_selection import train_test_split
assert not FORBIDDEN & set(NUM + CAT), "leaky or protected column in the feature list!"
tr, rest = train_test_split(df, test_size=0.4, stratify=df.churned_6m, random_state=RNG)
va, te = ____  # ✏️ B1
pd.DataFrame({"rows": [len(tr), len(va), len(te)], "churn_rate": [tr.churned_6m.mean(), va.churned_6m.mean(), te.churned_6m.mean()]},
             index=["train", "valid", "test"]).round(4)

**✔ Checkpoint.** 150k / 50k / 50k rows, each with a churn rate of 0.12.

## Step 2 · Algorithm bake-off on one validation set

**Why this matters.** Every algorithm gets the **same** features, training rows and validation rows, and is judged on the metrics the business uses: ROC-AUC (ranking), PR-AUC (imbalance-aware) and **capture in the top 10%** (the call budget). Fit time and scoring time matter too: k-NN has no training cost but must compare each new customer with every stored one.

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — churners caught ÷ all churners

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
def prep():
    return ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
def top_capture(y, p, budget=0.10):
    """share of all churners found when calling the top `budget` of customers"""
    top = np.argsort(-p)[: int(len(p) * budget)]
    return float(____)  # ✏️ B2
models = {"logistic": make_pipeline(prep(), LogisticRegression(max_iter=1000)),
          "k-NN (k=75)": make_pipeline(prep(), KNeighborsClassifier(n_neighbors=75, n_jobs=-1)),
          "decision tree": make_pipeline(prep(), DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=RNG)),
          "random forest": make_pipeline(prep(), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_samples=0.5, n_jobs=-1, random_state=RNG))}
X, Xv = tr[NUM + CAT], va[NUM + CAT]
board, P = {}, {}
for name, m in models.items():
    fit_rows = tr.sample(30_000, random_state=RNG) if name.startswith("k-NN") else tr
    t0 = time.time(); m.fit(fit_rows[NUM + CAT], fit_rows.churned_6m); t_fit = time.time() - t0
    t0 = time.time(); P[name] = p = m.predict_proba(Xv)[:, 1]; t_pred = time.time() - t0
    board[name] = {"ROC_AUC": roc_auc_score(va.churned_6m, p), "PR_AUC": average_precision_score(va.churned_6m, p),
                   "capture@10%": top_capture(va.churned_6m, p), "fit_s": t_fit, "predict_s": t_pred}
board = pd.DataFrame(board).T.sort_values("PR_AUC", ascending=False); board.round(3)

**✔ Checkpoint.** **Logistic regression wins** (ROC-AUC ≈ 0.71, PR-AUC ≈ 0.26, capture@10% ≈ 0.26) just ahead of the forest, which takes ≈ 10× longer to fit; the depth-8 tree follows, and k-NN is last *and* the slowest to predict. Churn is a noisy, hard target — AUC ≈ 0.7 is typical in banking. A random 10% call list would reach only 10% of churners.

## Step 3 · Depth controls over-fitting — a validation curve for trees

**Why this matters.** A tree grown without limits memorises the training set: training AUC → 1, validation AUC falls. **Depth** (and minimum leaf size, cost-complexity pruning) is the main lever. Plotting training *and* validation score against depth shows the under-fit → sweet spot → over-fit shape you met on Day 2.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — training score

In [ ]:
pre = prep().fit(X)
Xt, Xvt = pre.transform(X), pre.transform(Xv)
rows = []
for d in [2, 4, 6, 8, 10, 12, 16, 20, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=RNG).fit(Xt, tr.churned_6m)
    rows.append((str(d), ____,  # ✏️ B3
                 roc_auc_score(va.churned_6m, t.predict_proba(Xvt)[:, 1]), t.get_n_leaves()))
vc = pd.DataFrame(rows, columns=["max_depth", "train_AUC", "valid_AUC", "leaves"]).set_index("max_depth")
vc[["train_AUC", "valid_AUC"]].plot(marker="o", title="Decision tree: depth vs ROC-AUC"); plt.ylabel("ROC-AUC"); plt.show()
print("best depth on validation:", vc.valid_AUC.idxmax()); vc.round(3)

**✔ Checkpoint.** Validation AUC peaks at depth 6 (≈ 0.70) while training AUC keeps rising; the unlimited tree has ≈ 19k leaves, training AUC = 1.00 and validation AUC ≈ 0.53 — barely better than a coin. Textbook over-fitting.

## Step 4 · Gains, lift and the call list

**Why this matters.** A **cumulative-gains** chart answers the campaign question directly: *if we call the top x% by score, what share of all churners do we reach?* **Lift** = that share ÷ x. Decile tables are how retention and collections teams consume models.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — 1 = riskiest 10%

In [ ]:
champ = board.index[0]
dec = pd.DataFrame({"y": va.churned_6m.to_numpy(), "p": P[champ]})
dec["decile"] = ____  # ✏️ B4
table = dec.groupby("decile", observed=True).agg(customers=("y", "size"), churners=("y", "sum"), churn_rate=("y", "mean"))
table["lift"] = table.churn_rate / dec.y.mean()
table["cum_capture"] = table.churners.cumsum() / dec.y.sum()
display(table.round(3))
for name, p in P.items():
    order = np.argsort(-p); cum = np.cumsum(va.churned_6m.to_numpy()[order]) / va.churned_6m.sum()
    plt.plot(np.arange(1, len(p) + 1) / len(p), cum, label=name)
plt.plot([0, 1], [0, 1], "k--", label="random"); plt.axvline(0.10, color="grey", ls=":")
plt.xlabel("share of customers called"); plt.ylabel("share of churners reached"); plt.legend(); plt.title("Cumulative gains"); plt.show()

**✔ Checkpoint.** Decile 1 churn rate ≈ 31% (lift ≈ 2.6×); decile 10 ≈ 2%. The top 30% of customers hold ≈ 56% of all churners.

## Step 5 · Target by expected value, not by rank alone

**Why this matters.** Calling a customer costs AED 25 and a fee waiver AED 150 if accepted; ≈ 30% of would-be churners are saved, and a saved customer is worth ≈ 2% of their balance per year plus AED 400 product margin. **Expected value** = P(churn) × 0.30 × value − cost. Ranking by EV contacts high-value customers with moderate risk ahead of low-value customers with high risk.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — expected saving minus call and offer cost

In [ ]:
v = va.assign(p=P[champ], value=0.02 * np.expm1(va.log_balance) + 400)
v["ev"] = ____  # ✏️ B5
n10 = int(0.10 * len(v))
policies = {"top 10% by P(churn)": v.nlargest(n10, "p"), "top 10% by EV": v.nlargest(n10, "ev"), "all with EV > 0": v[v.ev > 0]}
pd.DataFrame({k: {"called": len(s), "churners_reached": int(s.churned_6m.sum()), "expected_value_AED": s.ev.sum()}
              for k, s in policies.items()}).T.round(0)

**✔ Checkpoint.** Top 10% by EV reaches fewer churners (≈ 990 vs ≈ 1,530) but more than **doubles** expected value (≈ AED 329k vs ≈ 140k). ‘All with EV > 0’ calls ≈ 17k customers (34%) for ≈ AED 445k — the most value, but over three times the call budget.

## Step 6 · Trees for regression — and why they cannot extrapolate

**Why this matters.** Trees also **regress**: each leaf predicts the mean of its training rows. That makes them flexible (no need for log transforms or splines) but **piecewise constant**: they can never predict outside the range seen in training. The Day 2 AVM was tested *out-of-time* on 2025 valuations, when prices kept rising. Can trees beat it?

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — bagged regression trees

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
val = pd.read_csv(DATA / "property_valuations.csv", parse_dates=["valuation_date"])
ratio = (val.valuation_aed / val.size_sqft) / (val.valuation_aed / val.size_sqft).groupby([val.community, val.property_type]).transform("median")
val = val[ratio.between(0.25, 4)].copy()
val["months_since_2018"] = (val.valuation_date - pd.Timestamp("2018-01-01")).dt.days / 30.44
VC = ["emirate", "community", "property_type", "view"]
VN = ["bedrooms", "size_sqft", "building_age_years", "floor", "furnished", "private_pool", "parking_spaces",
      "distance_to_metro_km", "service_charge_aed_sqft", "off_plan", "months_since_2018"]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
past, fut = val[val.valuation_date < "2025-01-01"], val[val.valuation_date >= "2025-01-01"]
Xp = np.c_[past[VN], enc.fit_transform(past[VC])]; Xf = np.c_[fut[VN], enc.transform(fut[VC])]
oot = lambda pred: np.mean(np.abs(np.exp(pred) / fut.valuation_aed - 1))
res = {f"tree depth {d}": oot(DecisionTreeRegressor(max_depth=d, min_samples_leaf=20, random_state=RNG).fit(Xp, np.log(past.valuation_aed)).predict(Xf))
       for d in [6, 12, 20]}
rf_reg = ____  # ✏️ B6
p_rf = rf_reg.fit(Xp, np.log(past.valuation_aed)).predict(Xf)
res["random forest"] = oot(p_rf)
res["Day 2 linear AVM (L2.1)"] = json.loads((CKPT / "valuation_metrics.json").read_text())["oot_2025_mape"]
display(pd.Series(res, name="2025 out-of-time MAPE").round(4).to_frame())
print(f"forest median bias on 2025: {np.median(np.exp(p_rf) / fut.valuation_aed - 1):+.1%}")

**✔ Checkpoint.** Depth 6 → 28% MAPE, depth 20 → 12.7%, forest → ≈ 10.8% — yet **every tree model loses to the Day 2 linear AVM (≈ 9.7%) out-of-time**, and the forest under-values 2025 properties (median bias ≈ −3%). A tree's prediction for `months_since_2018` beyond the training range is frozen at the last leaf — it cannot follow a rising market.

## Step 7 · Explain, test once, save the champion

**Why this matters.** Retention managers will ask *why* a customer is on the list. **Permutation importance** on validation shows which features the champion actually relies on. Then the test set — untouched so far — gives the number we report, and the scores are saved for the Day 4 neural challenger and the Day 5 capstone.

In [ ]:
import joblib
from sklearn.inspection import permutation_importance
m = models[champ]
vs = va.sample(15_000, random_state=RNG)
pi = permutation_importance(m, vs[NUM + CAT], vs.churned_6m, scoring="average_precision", n_repeats=3, random_state=RNG, n_jobs=-1)
pd.Series(pi.importances_mean, index=NUM + CAT).sort_values().tail(12).plot.barh(title=f"{champ}: permutation importance (PR-AUC drop)"); plt.show()
p_te = m.predict_proba(te[NUM + CAT])[:, 1]
test = {"ROC_AUC": roc_auc_score(te.churned_6m, p_te), "PR_AUC": average_precision_score(te.churned_6m, p_te), "capture@10%": top_capture(te.churned_6m, p_te)}
print("TEST", champ, {k: round(v, 3) for k, v in test.items()})
all_p = m.predict_proba(df[NUM + CAT])[:, 1]
split = pd.Series("train", index=df.index); split[va.index] = "valid"; split[te.index] = "test"
scores = pd.DataFrame({"customer_id": df.customer_id, "p_churn": all_p, "split": split,
                       "decile": pd.qcut(pd.Series(all_p).rank(method="first", ascending=False), 10, labels=range(1, 11)).astype(int).to_numpy()})
scores.to_parquet(CKPT / "churn_scores.parquet", index=False)
joblib.dump({"model": m, "name": champ, "features": NUM + CAT, "test": test, "valid": board.loc[champ].to_dict(),
             "split_ids": {s: df.customer_id[split == s].tolist() for s in ["train", "valid", "test"]}}, CKPT / "churn_model.joblib", compress=3)
print("saved churn_model.joblib and churn_scores.parquet")

**✔ Checkpoint.** Complaints, salary transfer, days since last transaction, digital logins, tenure and number of products dominate. Test metrics match validation (ROC-AUC ≈ 0.71, PR-AUC ≈ 0.25, capture@10% ≈ 0.26) — no sign of over-fitting to validation. Two files saved.

## ✅ Lab complete
Retention has a ranked, value-aware call list and Mortgage knows the limits of trees. **Tomorrow:** can SVMs and neural networks do better?